# Cellpose-SAM segmentation demo

Following **Pachitariu, Rariden, Stringer (2025)** — *Cellpose-SAM: superhuman generalization for cellular segmentation* ([bioRxiv 2025.04.28.651001](https://doi.org/10.1101/2025.04.28.651001)).

This notebook segments a multichannel fluorescence image with Cellpose-SAM and produces a per-cell quantification table. It is structured after the [MONAI spleen 3D tutorial](https://github.com/Project-MONAI/tutorials/blob/main/3d_segmentation/spleen_segmentation_3d.ipynb) but adapted for 2D fluorescence microscopy with a foundation-model segmenter.

## Why this is short

The paper's main point is that Cellpose-SAM is robust to channel order, cell size, noise, blur, and downsampling (their Fig 3). So the preprocessing pipeline in this notebook is short on purpose — `load → eval → quantify`. We deliberately skip:

- channel selection / DAPI-channel specification (the model is channel-order invariant — Fig 3a)
- diameter estimation (size-invariant within 7.5–120 px — Fig 3b)
- denoising / deblurring / upsampling (Fig 3c–f shows these aren't needed)

## Sections

1. Install + import
2. Load image
3. Build model
4. Segment
5. Per-cell quantification
6. Plots
7. Save outputs


## 1. Install and import

First run only — installs Cellpose 4.x (which provides the `cpsam` model) plus dependencies. The model weights (~370 MB) download automatically from Hugging Face the first time you call `CellposeModel(...)`.

In [ ]:
# Run once. Skip if already installed.
# %pip install --quiet "cellpose>=4.0" tifffile scikit-image pandas matplotlib
# For GPU (recommended; segmentation is ~10x faster):
# %pip install --quiet torch torchvision --index-url https://download.pytorch.org/whl/cu126

In [ ]:
import time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from PIL import Image
from skimage.measure import regionprops_table
from skimage.segmentation import find_boundaries
from cellpose import models
warnings.filterwarnings("ignore")
print("cellpose:", models.__name__)

## 2. Load the image

The included demo (`examples/fluorescence_sample.png`) is a multichannel fluorescence section: DAPI-stained nuclei (blue) plus marker channels in cyan/yellow/magenta/red. Replace `IMG_PATH` with your own file (TIFF, PNG, OME-TIFF, etc.).

In [ ]:
IMG_PATH = Path("examples/fluorescence_sample.png")
OUT_DIR  = Path("out");  OUT_DIR.mkdir(exist_ok=True)
NAME     = IMG_PATH.stem

arr = np.array(Image.open(IMG_PATH))
print(f"shape={arr.shape}, dtype={arr.dtype}, range=[{arr.min()}, {arr.max()}]")
plt.figure(figsize=(10, 8))
plt.imshow(arr)
plt.title(f"input — {arr.shape[1]} x {arr.shape[0]}")
plt.axis("off")
plt.show()

## 3. Build the Cellpose-SAM model

`pretrained_model="cpsam"` is the only Cellpose-SAM model. On a fresh install, this cell downloads ~370 MB the first time. After that the weights live in `~/.cellpose/models/`.

`gpu=True` will use CUDA (NVIDIA), MPS (Apple Silicon), or fall back to CPU. CPU on a large image is slow (minutes) — the paper reports ~0.4–1 s/image on an A100 and ~55 s on an RTX 4070 Super for similar-size images (their Table S1).

In [ ]:
t0 = time.time()
model = models.CellposeModel(gpu=True, pretrained_model="cpsam")
print(f"model loaded in {time.time() - t0:.1f}s")

## 4. Segment

Paper defaults (Methods § *Whole-dataset training and evaluation*):

- `flow_threshold=0.4`
- `cellprob_threshold=0.0`
- tile size 256, `tile_overlap=0.1`
- no test-time augmentation
- no image resizing — Cellpose-SAM runs natively at the input resolution

We pass the image as-is — no channel reordering, no nuclear-channel selection.

In [ ]:
t0 = time.time()
masks, flows, styles = model.eval(
    arr,
    batch_size=8,
    flow_threshold=0.4,
    cellprob_threshold=0.0,
    normalize=True,
    augment=False,
    tile_overlap=0.1,
)
print(f"segmented {int(masks.max())} cells in {time.time() - t0:.1f}s")

## 5. Per-cell quantification

For every segmented cell, `skimage.measure.regionprops_table` extracts:

- **morphology**: area, centroid, eccentricity, major/minor axis length, solidity
- **intensity** per channel: mean and max

This is the part that takes the place of MONAI's `DiceMetric` — in real-world fluorescence work you rarely have ground-truth masks for Dice; what you want is per-cell features you can gate, cluster, or plot.

In [ ]:
chan = np.moveaxis(arr, -1, 0) if arr.ndim == 3 else arr[None, ...]
df = pd.DataFrame(regionprops_table(
    masks, properties=("label", "area", "centroid",
                       "eccentricity", "major_axis_length",
                       "minor_axis_length", "solidity")))
for c, name in enumerate(["R", "G", "B"][:chan.shape[0]]):
    p = regionprops_table(masks, intensity_image=chan[c],
                          properties=("label", "intensity_mean", "intensity_max"))
    df[f"{name}_mean"] = p["intensity_mean"]
    df[f"{name}_max"]  = p["intensity_max"]
df.insert(0, "image", IMG_PATH.name)
print(f"{len(df)} cells quantified")
df.head()

In [ ]:
df.describe().round(2)

## 6. Plots

Three-panel overlay: the raw image, the colored instance mask, and red cell boundaries painted on the input.

In [ ]:
i2 = arr.astype(np.float32) / arr.max()
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(i2);                          ax[0].set_title("input")
ax[1].imshow(masks, cmap="nipy_spectral"); ax[1].set_title(f"masks  (n={int(masks.max())})")
ov = i2.copy()
ov[find_boundaries(masks, mode="outer")] = [1.0, 0.2, 0.2]
ax[2].imshow(ov);                          ax[2].set_title("overlay")
for a in ax: a.axis("off")
fig.tight_layout()
plt.show()

Per-channel intensity histograms across all segmented cells — useful for gating marker-positive populations.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for c, (name, color) in enumerate(zip(["R", "G", "B"], ["red", "green", "blue"])):
    if f"{name}_mean" in df.columns:
        axes[c].hist(df[f"{name}_mean"], bins=40, color=color, alpha=0.7, edgecolor="k")
        axes[c].set_title(f"{name} channel — per-cell mean")
        axes[c].set_xlabel("mean intensity")
        axes[c].set_ylabel("# cells")
fig.tight_layout()
plt.show()

## 7. Save outputs

Three files per image:

- `{name}_masks.tif` — int32 instance mask, opens in Fiji / napari / QuPath
- `{name}_cells.csv` — per-cell feature table
- `{name}_overlay.png` — the three-panel figure above

`all_cells.csv` would combine results across multiple images in a batch run.

In [ ]:
tifffile.imwrite(str(OUT_DIR / f"{NAME}_masks.tif"), masks.astype(np.int32))
df.to_csv(OUT_DIR / f"{NAME}_cells.csv", index=False)

# Save the overlay as a standalone file
fig, ax = plt.subplots(1, 3, figsize=(18, 6))
ax[0].imshow(i2);                          ax[0].set_title("input")
ax[1].imshow(masks, cmap="nipy_spectral"); ax[1].set_title(f"masks  (n={int(masks.max())})")
ax[2].imshow(ov);                          ax[2].set_title("overlay")
for a in ax: a.axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / f"{NAME}_overlay.png", dpi=120, bbox_inches="tight")
plt.close(fig)

print("wrote:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f}  ({f.stat().st_size:,} bytes)")

## Next steps

- **Batch processing**: see `cellpose_sam.py` for a CLI version that processes a folder and (optionally) commits results to GitHub via `gh`.
- **Fine-tuning**: Cellpose-SAM is a foundation model; for very different tissue types you can fine-tune on a small labeled set (10–500 ROIs is often enough — see Fig 4 of the paper). The [official `train_Cellpose-SAM.ipynb`](https://github.com/MouseLand/cellpose/blob/main/notebooks/train_Cellpose-SAM.ipynb) shows how.
- **3D**: pass `do_3D=True` and an `(Z, Y, X)` or `(Z, C, Y, X)` array. The CLI script in this repo auto-detects 3D.

## Citation

If you use this in published work, cite Cellpose-SAM:

> Pachitariu M, Rariden M, Stringer C. *Cellpose-SAM: superhuman generalization for cellular segmentation.* bioRxiv 2025.04.28.651001 (2025).
